# Chapter 17: Forty thousand messages

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "17"
__file__ = os.path.abspath("run.py")

## The scene

Headquarters sends a synthetic batch to test the classifier before it goes to every force:
invented messages in the shape of real intercepts, with the right label on each. Forty thousand
a month is the figure. "Last time a system like this went live," Varma says, "it ran beautifully
for an hour and then the first error stopped the whole queue. Nobody noticed until morning."

She wants three numbers before anything is switched on: how many messages a minute LANTERN can
sort, how long the slowest ones take, and what happens when calls start to fail.

The batch is made by [`generate.py`](generate.py), seeded, so every run sorts the same messages.

## The concept

Forty-eight messages hide every problem this chapter is about. At a few thousand, the problems
are the point:

- **Throughput**: messages per minute. It decides whether a month's work takes an afternoon or a
  week.
- **Tail latency**: the 95th percentile (p95), the time that 95 messages in 100 finish within.
  The median says what a typical message costs; the tail says what the queue behind it feels.
- **Failures**: at scale, some calls fail. A rate limit (HTTP 429), an overloaded server (503), a
  call that never answers. One unhandled failure must not stop the batch, and a thousand retries
  must not turn a short outage into a long one.

![Batch runner](../../docs/diagrams/17-batch-runner.svg)

The runner in the diagram is the chapter's exercise. Five parts, each answering one failure:

| Part | Without it |
|---|---|
| Bounded concurrency | Either one call at a time, or as many as the server will queue |
| A timeout per attempt | One stuck call holds a worker for minutes |
| Retries on `Transient` only | A blip fails a message; or a bad request is sent again and again |
| Exponential backoff with full jitter | Every worker that failed together retries together |
| A retry budget for the batch | An outage multiplies the load on the service that is failing |

## How it works

**The batch.** `generate.messages(n)` fills thirty templates (logistics, personal, threat) with
seeded details: ships, piers, invoice numbers, days. Each message has a gold label, so the batch
also measures agreement.

**One attempt.** `chat(..., retries=0, timeout=3)`: the runner owns retries, so the client's own
are off. `lantern.chat` measures every call on the wall clock, logs it to `.progress/calls.jsonl`
with its prompt version (`classify-message@v1`), model, tokens, cost and latency
(`lantern.calllog`), and traces it as an OpenTelemetry span with GenAI attributes in
`.progress/traces.jsonl` (`lantern.trace`).

**Retries.** `lantern.retry.backoff(n, base, cap)` is full jitter: a random wait between 0 and
`min(cap, base * 2**n)`. With `base=0.25` and `cap=4`, the first retry waits up to 0.25 s, the
third up to 1 s. A `retry-after` hint from the provider is a floor on the wait.

**Faults, on purpose.** `Faults` wraps the model call. For a seeded share of attempts it raises
`Transient("HTTP 429")` or `Transient("HTTP 503")` instead of calling the model, or it hangs until
the timeout and then fails, as httpx does when a server never answers. The draw depends only on
the message and the attempt number, so every run injects the same faults.

**Reading it back.** Each batch runs inside `trace.span(...)`. Worker threads do not inherit the
trace context, so the chapter carries it over (`in_this_trace`) and each chat span nests under its
batch. The numbers in steps 2, 3 and 5 are read back from the call log, not kept in memory: the
log is what an operator would have.

```python
results = run_batch(batch, chat=faulty, prompt=prompt, concurrency=2, timeout=3.0,
                    retries=3, base=0.25, cap=4.0, retry_budget=len(batch) // 5)
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import argparse
import random
import sys
import threading
import time
from pathlib import Path

from opentelemetry import context as otel_context

sys.path.insert(0, str(Path(__file__).resolve().parent))
import generate  # noqa: E402  (the seeded batch, beside this file)

from lantern import calllog, chat, learn, trace  # noqa: E402
from lantern.config import load_config, model_name, tier  # noqa: E402
from lantern.llm import Reply, cost_of  # noqa: E402
from lantern.retry import Transient  # noqa: E402
from lantern.ui import (  # noqa: E402
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    save_shot,
    step,
    your_turn,
)

local = tier() == "local"
ap = argparse.ArgumentParser()
ap.add_argument("--batch", type=int, default=1200 if local else 120,
                help="messages in the main batch (smaller on the hosted tier: it is billed)")
ap.add_argument("--sweep", type=int, default=300 if local else 30,
                help="messages per concurrency level in the sweep")
ap.add_argument("--concurrency", type=int, default=2, help="concurrency of the main batch")
args, _ = ap.parse_known_args()

chapter("17", "Forty thousand messages", """
Headquarters sends a synthetic batch to test the classifier before it goes to every force:
invented messages in the shape of real intercepts, with the right label on each. Forty thousand
a month is the figure. "Last time a system like this went live," Varma says, "it ran beautifully
for an hour and then the first error stopped the whole queue. Nobody noticed until morning."

She wants three numbers before anything is switched on: how many messages a minute LANTERN can
sort, how long the slowest ones take, and what happens when calls start to fail.
""")

MINE = learn.mine()
run_batch = MINE.run_batch if MINE else None
if run_batch is None:
    from batch import run_batch  # noqa: E402  (the lesson's runner; --mine uses yours)

MODEL = model_name("small")
VERSION = "classify-message@v1"
LABELS = ["threat", "logistics", "personal"]
SYSTEM = """Classify a text message from a company executive's work phone.
Labels:
- threat: intimidation, pressure or warnings aimed at the recipient
- logistics: work operations, schedules, deliveries, invoices
- personal: family, friends, health, private life
Answer with exactly one word: threat, logistics or personal."""
TIMEOUT = 3.0  # seconds for one attempt; step 1 shows why
RETRIES, BASE, CAP = 3, 0.25, 4.0

batch = generate.messages(args.batch)
evidence(f"synthetic batch: {len(batch):,} messages, seed {generate.SEED} (first 3)", "\n".join(
    f"{m['id']}  {m['time']}  {m['from']:<18} {m['text']}" for m in batch[:3]))


def prompt(m: dict) -> list[dict]:
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Message from {m['from']}: \"{m['text']}\""}]


def parse(text: str) -> str:
    for word in text.lower().replace(".", " ").replace(":", " ").split():
        if word in LABELS:
            return word
    return "invalid"


def model_call(messages: list[dict], **kw) -> Reply:
    """One attempt, no retries inside: the batch runner owns retries. lantern.chat turns a
    Transient it will not retry into SystemExit; give the runner back the Transient."""
    try:
        return chat(messages, model=MODEL, temperature=0, max_tokens=6, retries=0,
                    prompt_version=VERSION, **kw)
    except SystemExit as e:
        if isinstance(e.__cause__, Transient):
            raise e.__cause__ from None
        raise


def in_this_trace(fn):
    """Worker threads do not inherit the trace context: carry it over, so each chat span nests
    under the batch span."""
    ctx = otel_context.get_current()

    def call(messages, **kw):
        token = otel_context.attach(ctx)
        try:
            return fn(messages, **kw)
        finally:
            otel_context.detach(token)

    return call


class Faults:
    """Seeded fault injection: a set share of attempts fail as a 429, a 503, or a stuck call.

    The draw depends only on the message and the attempt number, so the same faults hit the same
    calls whatever order the threads run in. A stuck call hangs until the caller's timeout and
    then fails, which is what httpx does when a server never answers.
    """

    def __init__(self, fn, *, r429: float, r503: float, stuck: float, seed: int = 17):
        self.fn, self.r429, self.r503, self.stuck, self.seed = fn, r429, r503, stuck, seed
        self.lock = threading.Lock()
        self.tries: dict[str, int] = {}
        self.count = {"429": 0, "503": 0, "stuck": 0}

    def __call__(self, messages, *, timeout: float, **kw):
        key = messages[-1]["content"]
        with self.lock:
            n = self.tries[key] = self.tries.get(key, 0) + 1
        x = random.Random(f"{self.seed}:{key}:{n}").random()
        if x < self.r429:
            kind = "429"
        elif x < self.r429 + self.r503:
            kind = "503"
        elif x < self.r429 + self.r503 + self.stuck:
            kind = "stuck"
        else:
            return self.fn(messages, timeout=timeout, **kw)
        with self.lock:
            self.count[kind] += 1
        if kind == "stuck":
            time.sleep(timeout)
            raise Transient(f"no answer within {timeout:.0f}s (injected stuck call)")
        raise Transient(f"HTTP {kind} (injected)")


def pct(values: list[float], p: float) -> float:
    """Nearest-rank percentile: p=50 is the median, p=95 the value 95% of calls are under."""
    if not values:
        return 0.0
    s = sorted(values)
    return s[max(0, min(len(s) - 1, round(p / 100 * len(s) + 0.5) - 1))]


def logged_since(mark: int) -> list[dict]:
    """This chapter's call-log records written after `mark` lines."""
    return [r for r in calllog.read()[mark:] if r.get("chapter") == "17"
            and r.get("lantern.prompt.version") == VERSION]


def go(items, *, concurrency: int, fn, name: str, **kw) -> dict:
    """Run one batch inside a span; read its numbers back from the call log."""
    mark = len(calllog.read())
    events: list[dict] = []
    with trace.span(name, **{"lantern.batch.size": len(items),
                             "lantern.batch.concurrency": concurrency}) as sp:
        start = time.perf_counter()
        results = run_batch(items, chat=in_this_trace(fn), prompt=prompt,
                            concurrency=concurrency, timeout=TIMEOUT, retries=kw.pop("retries",
                            RETRIES), base=BASE, cap=CAP, log=events.append, **kw)
        wall = time.perf_counter() - start
        trace_id = f"0x{sp.get_span_context().trace_id:032x}"
    log = logged_since(mark)
    from_log = len(log) == sum(r["ok"] for r in results) and bool(log)
    call_secs = ([r["lantern.latency_s"] for r in log] if from_log
                 else [r["reply"].seconds for r in results if r["ok"]])
    ok = [r for r in results if r["ok"]]
    correct = sum(parse(r["reply"].text) == m["label"]
                  for r, m in zip(results, items, strict=True) if r["ok"])
    return {
        "results": results, "events": events, "wall": wall, "log": log, "from_log": from_log,
        "n": len(items), "ok": len(ok), "failed": len(items) - len(ok),
        "per_min": len(ok) / wall * 60 if wall else 0.0,
        "p50": pct(call_secs, 50), "p95": pct(call_secs, 95),
        "item_p50": pct([r["seconds"] for r in ok], 50),
        "item_p95": pct([r["seconds"] for r in ok], 95),
        "retries": sum(e["outcome"] == "retry" for e in events),
        "retried_items": sum(r["attempts"] > 1 for r in results),
        "correct": correct, "trace_id": trace_id,
    }

### 1. Warm up, and choose the timeout

In [ ]:
step("1. Warm up, and choose the timeout")
warm = [model_call(prompt(m), timeout=60) for m in batch[:20]]
warm_secs = [r.seconds for r in warm[1:]]
note(f"{MODEL}: the first call took {warm[0].seconds:.2f}s (it may load the model); the next "
     f"{len(warm_secs)} took {pct(warm_secs, 50):.2f}s at the median and {max(warm_secs):.2f}s at "
     "most.")
note(f"Timeout per attempt: {TIMEOUT:.0f}s. Long enough that a healthy call is never cut off, "
     "short enough that a stuck one does not hold a worker for minutes. config.yaml's 300s is "
     "for a cold 4B model on a CPU, not for a warm batch.")

### 2. Throughput at concurrency 1, 2 and 4 ({args.sweep} messages each, no faults)

In [ ]:
step(f"2. Throughput at concurrency 1, 2 and 4 ({args.sweep} messages each, no faults)")
p_conc = learn.predict(
    "Going from one call at a time to four at once, how will messages per minute change?",
    ["about 4 times as many", "about 2 times as many", "about the same", "fewer"])
sweep_items = batch[: args.sweep]
sweep = {c: go(sweep_items, concurrency=c, fn=model_call, name=f"sweep c={c}") for c in (1, 2, 4)}
s = Table(title=f"{MODEL}, {args.sweep} messages per row, read back from .progress/calls.jsonl")
for col in ("concurrency", "messages/min", "call p50 s", "call p95 s", "failed", "correct"):
    s.add_column(col, justify="right")
for c, res in sweep.items():
    s.add_row(str(c), f"{res['per_min']:,.0f}", f"{res['p50']:.3f}", f"{res['p95']:.3f}",
              str(res["failed"]), f"{res['correct']}/{res['n']}")
console.print(s)
gain = sweep[4]["per_min"] / sweep[1]["per_min"] if sweep[1]["per_min"] else 0.0
learn.reveal(p_conc, answer=learn.bucket(-gain, [-3.0, -1.5, -0.9]),
             why=f"Four at once gave {gain:.2f} times the throughput of one. "
             + ("Ollama here runs one request at a time (one parallel slot): extra concurrency "
                "only queues at the server, so each call's latency grows while throughput "
                "barely moves. Concurrency helps only up to the server's own parallelism."
                if local else "A hosted API serves many requests at once, up to your rate "
                "limit, so concurrency is where its throughput comes from."))
if not all(r["from_log"] for r in sweep.values()):
    note("The call log is off (LANTERN_CALL_LOG is empty): these numbers come from the replies.")

### 3. The whole batch, {len(batch):,} messages at concurrency {args.concurrency}, with faults

In [ ]:
step(f"3. The whole batch, {len(batch):,} messages at concurrency {args.concurrency}, with faults")
FAULTS = {"r429": 0.04, "r503": 0.03, "stuck": 0.01}
note(f"Injected, seeded: {FAULTS['r429']:.0%} of attempts get a 429, {FAULTS['r503']:.0%} a 503, "
     f"{FAULTS['stuck']:.0%} hang until the {TIMEOUT:.0f}s timeout cuts them off.")
p_fail = learn.predict(
    f"With {RETRIES} retries, exponential backoff and full jitter, how many of the "
    f"{len(batch):,} messages will still fail?",
    ["none", "1 to 10", "11 to 50", "more than 50"])
faulty = Faults(model_call, **FAULTS)
budget = max(1, len(batch) // 5)
main = go(batch, concurrency=args.concurrency, fn=faulty, name="batch with faults",
          retry_budget=budget)
m = Table(title=f"{len(batch):,} messages, {MODEL}, concurrency {args.concurrency}, timeout "
          f"{TIMEOUT:.0f}s, {RETRIES} retries, budget {budget}")
for col in ("", "count"):
    m.add_column(col, justify="left" if not col else "right")
injected = sum(faulty.count.values())
m.add_row("faults injected (429 / 503 / stuck)",
          f"{injected} ({faulty.count['429']} / {faulty.count['503']} / {faulty.count['stuck']})")
m.add_row("messages that needed a retry", str(main["retried_items"]))
m.add_row("retries spent (of the budget)", f"{main['retries']} of {budget}")
m.add_row("messages that failed in the end", str(main["failed"]))
m.add_row("messages per minute", f"{main['per_min']:,.0f}")
m.add_row("model call p50 / p95 s (from the call log)", f"{main['p50']:.3f} / {main['p95']:.3f}")
m.add_row("whole message p50 / p95 s (waits included)",
          f"{main['item_p50']:.3f} / {main['item_p95']:.3f}")
m.add_row("agreement with the gold labels", f"{main['correct']}/{len(batch):,}")
console.print(m)
learn.reveal(p_fail, answer=learn.bucket(main["failed"], [0, 10, 50]),
             why=f"{main['failed']} failed. {main['retried_items']} messages hit at least one "
             "fault; without retries every one of them would have failed. A fault has to strike "
             f"the same message {RETRIES + 1} times in a row to beat the retries.")
note("The faults show in the tail, not the middle: the median message barely moved, the 95th "
     "percentile carries the backoff waits and the cut-off stuck calls.")

### 4. An outage: the retry budget

In [ ]:
step("4. An outage: the retry budget")
p_budget = learn.predict(
    "Now 70% of attempts fail, as in a real outage. With a retry budget of 20 for 100 messages, "
    "how many attempts will the batch make in total?",
    ["about 100", "about 120", "about 250", "about 400"])
outage = Faults(model_call, r429=0.4, r503=0.3, stuck=0.0, seed=4)
storm = go(batch[:100], concurrency=args.concurrency, fn=outage, name="outage with budget",
           retry_budget=20)
attempts = sum(r["attempts"] for r in storm["results"])
learn.reveal(p_budget, answer=learn.bucket(attempts, [110, 180, 320]),
             why=f"{attempts} attempts, {storm['failed']} messages failed. Without a budget the "
             f"same outage could make up to {100 * (RETRIES + 1)}: every retry is more load on a "
             "service that is already failing. The budget turns an outage into a short list of "
             "messages to run again later, instead of a retry storm.")
note(f"70% of attempts failing, 100 messages, retry budget 20: {attempts} attempts in all, "
     f"{storm['failed']} messages failed and are listed to run again. Without the budget the same "
     f"outage could have made up to {100 * (RETRIES + 1)} attempts on a service already failing.")

### 5. What the log says

In [ ]:
step("5. What the log says")
logged = main["log"] + [r for res in sweep.values() for r in res["log"]] + storm["log"]
tokens_in = sum(r["gen_ai.usage.input_tokens"] for r in logged)
tokens_out = sum(r["gen_ai.usage.output_tokens"] for r in logged)
spans = 0
if trace.PATH.exists():
    for line in trace.PATH.read_text(encoding="utf-8").splitlines():
        if main["trace_id"] in line and '"chat ' in line:
            spans += 1
per_msg_in = tokens_in / len(logged) if logged else 0
per_msg_out = tokens_out / len(logged) if logged else 0
hosted_model = load_config()["hosted"]["anthropic"]["models"]["small"]
est = cost_of(Reply("", hosted_model, provider="anthropic", prompt_tokens=round(per_msg_in),
                    output_tokens=round(per_msg_out)))
month_minutes = 40_000 / main["per_min"] if main["per_min"] else 0.0
g = Table(title="Read back from .progress/calls.jsonl and .progress/traces.jsonl")
for col in ("", "value"):
    g.add_column(col, justify="left" if not col else "right")
g.add_row("calls logged this run (prompt version " + VERSION + ")", f"{len(logged):,}")
g.add_row("input / output tokens", f"{tokens_in:,} / {tokens_out:,}")
g.add_row("cost logged (USD)", f"{sum(r['lantern.cost_usd'] or 0 for r in logged):.4f}")
g.add_row("chat spans under the 'batch with faults' trace", f"{spans:,}")
g.add_row(f"40,000 messages at {main['per_min']:,.0f}/min (extrapolated)",
          f"{month_minutes / 60:.1f} hours")
g.add_row(f"40,000 messages on {hosted_model}, estimated from these token counts",
          f"${est * 40_000:.2f}" if est is not None else "unpriced")
console.print(g)
note("Only successful calls reach the call log: a failed attempt never returns a reply. The "
     "runner's own log events count the failures; the call log counts the work done.")

learn.record(
    model=MODEL,
    batch=len(batch),
    sweep=args.sweep,
    concurrency=args.concurrency,
    per_min_c1=round(sweep[1]["per_min"], 1),
    per_min_c2=round(sweep[2]["per_min"], 1),
    per_min_c4=round(sweep[4]["per_min"], 1),
    p50_c1=round(sweep[1]["p50"], 3),
    p95_c1=round(sweep[1]["p95"], 3),
    p95_c4=round(sweep[4]["p95"], 3),
    faults_injected=injected,
    retried_items=main["retried_items"],
    retries_spent=main["retries"],
    failed=main["failed"],
    per_min_batch=round(main["per_min"], 1),
    call_p95_batch=round(main["p95"], 3),
    item_p95_batch=round(main["item_p95"], 3),
    correct=main["correct"],
    outage_attempts=attempts,
    outage_failed=storm["failed"],
    logged_calls=len(logged),
    spans=spans,
    mine=bool(MINE),
)

your_turn(
    "17",
    check=[
        "In step 2, compare call p95 at concurrency 1 and 4. Where did the extra time go?",
        "In step 3, compare the model call p95 with the whole message p95. What is in the gap?",
        "In step 4, how many messages would you run again tomorrow, and how do you find them?",
    ],
    exercises=[
        "In run.py, set RETRIES to 0 and rerun. How many messages fail in step 3?",
        "In run.py, set BASE to 0 (no backoff) and give the outage a budget of 1,000. Count the "
        "attempts in step 4.",
        "In run.py, set TIMEOUT to 0.05. What happens to healthy calls?",
        "No edit: make lesson N=17 ARGS=\"--concurrency 4\". Does the batch finish sooner?",
    ],
)

right_share = main["correct"] / len(batch)
verdict = ("wrong most of the time" if right_share < 0.5 else
           "wrong more often than a court would accept" if right_share < 0.9 else
           "right nine times in ten or better")
finding(f"""
On this machine LANTERN sorts about {main['per_min']:,.0f} messages a minute on {MODEL}, with
faults, so a month of forty thousand is about {month_minutes / 60:.1f} hours of work. Of {injected}
attempts made to fail on purpose, {main['failed']} messages were lost, and the budget held an outage
to {attempts} attempts. Every call is in the log with its prompt version, model, tokens and latency.

The answers are another matter: {main['correct']:,} of {len(batch):,} agree with the labels.
Varma reads the log, not the summary. "Every call, with its version and its time," she says.
"Good. Hale will ask for exactly that. And he will ask why we ran a model that is {verdict}.
Before anything goes to court, we prove the pipeline is right, not just that it runs."
""")
save_shot("17-reliable-calls-at-scale")

## The nuances

**1. Concurrency helps only up to the server's parallelism.** One call at a time sorted 1,456
messages a minute; two at a time 1,800; four at a time 1,809. Past two, the extra workers only
waited: Ollama runs a fixed number of requests at once and queues the rest, so the median call got
slower (0.040 s, then 0.065 s, then 0.131 s) while throughput stayed flat. Measure where the curve
bends before you size a worker pool.

**2. Report the tail.** In the faulted batch the median message took 0.049 s, almost the same as
with no faults. The 95th percentile took 0.198 s: that is where the backoff waits and the cut-off
stuck calls went. A dashboard that shows only the median shows a healthy service during an
incident.

**3. Retry only what can succeed next time.** A 429, a 5xx, a timeout or a dropped connection may
succeed a moment later. A 400 (a schema the provider rejects, a prompt too long) fails the same
way every time; retrying it burns quota. `lantern.retry.Transient` marks the first kind, and the
runner fails the second at once.

**4. Jitter is what makes backoff work.** Plain exponential backoff sends every client that failed
together back together: 1 s, 2 s, 4 s, in step, and the second wave fails like the first. Full
jitter waits a random time between zero and the exponential cap, which spreads the retries out.
A provider's `retry-after` is a floor on that wait, never ignored.

**5. A budget turns an outage into a list.** With 70% of attempts failing and a budget of 20
retries for 100 messages, the batch made 120 attempts and listed 61 messages to run again. Without
the budget the same outage could have made up to 400 attempts against a service that was already
failing. Retries are load; a budget caps it.

**6. Set the timeout from the measured tail.** Warm, qwen3:0.6b answered in 0.04 s at the median
and at most; its first call took 1.47 s because it loaded the model. The batch uses 3 s per
attempt: well above any healthy call, so none is cut off, and short enough that a stuck one frees
its worker in seconds. `config.yaml`'s 300 s is for a cold 4B model on a CPU, not a warm batch.

**7. Only successful calls reach the call log.** A failed attempt never produces a reply, so the
call log counts the work done and the runner's own events count the failures. Read both. Every
logged call carries its prompt version, so a change of prompt shows up as a new version in the
log, not as a mystery in the numbers.

**8. Fast is not the same as right.** At this scale the plumbing held: 98 injected faults, 0
messages lost. The classifier did not: qwen3:0.6b agreed with the gold labels on 351 of 1,200
messages (29%), no better than guessing among three labels, because it calls most messages a
threat. A throughput number is only worth having for a model whose answers you have already
measured (chapter 12). And extrapolate with care: 1,158 a minute makes forty thousand messages
about 0.6 hours, if nothing else on the machine changes.

## On a hosted model

```bash
make lesson N=17 TIER=hosted            # needs ANTHROPIC_API_KEY in .env; 120 messages by default
make lesson N=17 TIER=hosted REPLAY=1   # from a recorded run, once one is committed
```

On the hosted tier the batch is smaller by default (`--batch 120 --sweep 30`): every call is
billed. The model is `claude-haiku-5-5`. The runner is the same; what differs is the server. A
hosted API runs many requests at once, up to the account's rate limit, so concurrency is where its
throughput comes from, and a real 429 carries a `retry-after` header that `lantern.cloud` passes
to the runner as the floor on its wait. The Anthropic SDK's own retries are turned off
(`max_retries=0`) so that every attempt is counted here. Claude does not take `temperature` or
`seed`; they are not sent and are reported once on the console.

| Measure | Local: qwen3:0.6b, Ollama, Apple M1 Pro 16 GB | Hosted: claude-haiku-5-5 |
|---|---|---|
| messages per minute, concurrency 1 | 1,456 | not run |
| messages per minute, concurrency 4 | 1,809 | not run |
| whole message p95 s, faulted batch | 0.198 | not run |
| agreement with the gold labels | 351 of 1,200 | not run |
| cost logged, USD | 0.0000 | not run |

Live hosted numbers are not run: this lab has no hosted key, and no number here is estimated.

What the hosted run will test: whether throughput keeps rising with concurrency on a server that
runs many requests at once (the flat curve past two is likely an Ollama effect), what real 429s
and `retry-after` waits look like at the account's rate limit, and whether a larger model's
agreement with the labels makes the throughput worth having. That retries recover transient
faults, that a budget caps an outage and that the tail carries the faults should hold on both.

## Try this

1. Set `RETRIES = 0` and rerun. How many messages fail in step 3, and which faults caused them?
2. Set `BASE = 0` and give the outage a budget of 1,000. Count the attempts in step 4.
3. Set `TIMEOUT = 0.05`. What happens to healthy calls, and to the retry budget?
4. Run `make lesson N=17 ARGS="--concurrency 4"`. Does the batch finish sooner? Compare the call
   p95 with step 2.
5. Open `.progress/calls.jsonl` and find the slowest call of the run. Then find its span in
   `.progress/traces.jsonl` by its time, and read which batch it belonged to.

## Master it

After this chapter you can measure a batch in messages per minute and p50 and p95 latency, read
the numbers back from a call log, say why concurrency stops helping, and write a runner that
survives rate limits, server errors and stuck calls without turning an outage into a storm.

**Predict, then see.** `make lesson N=17` stops three times to ask: how throughput changes from
one call at a time to four, how many messages still fail after retries, and how many attempts an
outage costs with a retry budget. Each answer is checked against your own run. Add
`ARGS=--no-predict` to watch without stopping.

**Build it.** Write `run_batch()` in [`practice/exercise.py`](practice/exercise.py): bounded
concurrency, a timeout, retries with backoff and full jitter, a retry budget and a log event per
attempt. run.py uses it in place of its own with `--mine`.

```bash
make exercise N=17             # 9 offline tests, scripted model, no Ollama needed
make hint N=17                 # one hint at a time
make solution N=17             # the reference, once you have tried
make lesson N=17 ARGS=--mine   # the whole batch, faults and all, on your runner
```

**Check yourself.** `make quiz N=17`: six questions, each with the reason behind the answer.

**Break it.** `make challenge N=17` checks your latest run against three targets: your runner on
the real batch, losing no more than five messages; no retries, so the faults get through; and a
retry storm in the outage, three times the attempts the budget allowed.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 17

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 17

In [ ]:
%run ../../scripts/learn.py hint 17

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 17

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

On this machine LANTERN sorts about 1,158 messages a minute on qwen3:0.6b, with faults, so a month
of forty thousand is about 0.6 hours of work. Of 98 attempts made to fail on purpose, every one was
retried and recovered, and the budget held an outage to 120 attempts. Every call is in the log
with its prompt version, model, tokens and latency, and in the trace under its batch.

The answers are another matter: 351 of 1,200 agree with the labels. Varma reads the log, not the
summary. "Every call, with its version and its time," she says. "Good. Hale will ask for exactly
that. And he will ask why we ran forty thousand messages through a model that is wrong two times
in three. Before anything goes to court, we prove the pipeline is right, not just that it runs."

**Next:** [Chapter 18: Proven, not demonstrated](../18-evals-with-promptfoo/README.md)